# TB Portals - LOCAL 04 - Eval / GATE verification

Aggregates results.csv and compares against Kantipudi A2.
**Gate passes** when Timika-MAE delta <= 3.0 AND Pearson delta >= -0.1 per held-out country.

In [ ]:
# ── Paths (must match local_01_build_manifest) ──
import os, sys
from pathlib import Path
REPO_DIR = str(Path(os.path.abspath("")).parent)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
WORK     = str(Path(REPO_DIR) / "local_work")
MANIFEST = f"{WORK}/data/processed/tbportals_manifest.csv"
OUT_DIR  = f"{WORK}/checkpoints/tbportals/baseline"
print("REPO_DIR:", REPO_DIR)


In [ ]:
import pandas as pd, numpy as np
from src.evaluation.eval_tbportals import KANTIPUDI_A2

res = pd.read_csv(f"{OUT_DIR}/results.csv")
res.head()


In [ ]:
rows = []
for country in ["Romania", "Moldova", "Kazakhstan"]:
    sub = res[res.held_out == country]
    if len(sub) == 0:
        continue
    k = KANTIPUDI_A2[country]
    rows.append({
        "country": country, "n_runs": len(sub),
        "timika_mae":    f"{sub.timika_mae.mean():.2f}+/-{sub.timika_mae.std(ddof=0):.2f}",
        "timika_mae_K":  k["timika_mae"],
        "timika_pearson":f"{sub.timika_pearson.mean():.2f}",
        "pearson_K":     k["timika_pearson"],
        "alp_mae":       f"{sub.alp_mae.mean():.2f}",  "alp_mae_K": k["alp_mae"],
        "cavity_auc":    f"{sub.cavity_auc.mean():.2f}","cavity_auc_K": k["cavity_auc"],
    })
pd.DataFrame(rows)


In [ ]:
# Programmatic gate verdict
ok = True
for country in ["Romania", "Moldova", "Kazakhstan"]:
    sub = res[res.held_out == country]
    if len(sub) == 0:
        print(f"{country}: NO RUNS"); ok = False; continue
    k = KANTIPUDI_A2[country]
    dmae  = sub.timika_mae.mean()     - k["timika_mae"]
    dpear = sub.timika_pearson.mean() - k["timika_pearson"]
    verdict = "PASS" if (dmae <= 3.0 and dpear >= -0.1) else "REVIEW"
    if verdict != "PASS": ok = False
    print(f"{country}: Timika MAE delta={dmae:+.2f}, Pearson delta={dpear:+.2f} -> {verdict}")
print("
DAY-1 GATE:", "PASSED - proceed to Day 2" if ok else "REVIEW - check ALP scale / country join / patient leakage")
